# Notebook 06 — License Plate Recognition

**Objective.** Read the plate of a tracked vehicle, say how far that reading can be trusted, and say so plainly when it cannot be read.

**Background.** A plate links a candidate violation to a vehicle, so a wrong plate could point at the wrong person. That makes a confident wrong text worse than no text. Every result here supports a person's review and decides nothing.

**Input.**
- `data/traffic_video_modified.mp4` — **development**: night traffic in which near plates can be read by eye. Its signal is a graphic added to the footage, which section 7 treats as the real signal, as a simulation.
- `data/video-test.mp4` — **evaluation**: daytime traffic with different plates, including two-line plates on motorcycles. Settings were committed before this video was run with them.
- `outputs/05b_red_light_violation/rule_v2/rlr_occasions_violation_clips/manifest.json` — the clips of the candidate violations from Notebook 05b, with rule version 2, the route real cases will take.
- `data/private_annotations/plate_labels.json` — labels made by eye. It holds real plate numbers, so it is git-ignored, and the scoring is skipped where it is missing.

**Output**, in `outputs/06_license_plate_recognition/` (git-ignored):
- `<video name>/`: `<video name>_output.mp4`, `<video name>_plate_crops/`, `<video name>_plates.json`.
- `violation_clips/`: one video per event, `plate_crops/`, and `plates.json` with a `license_plate` result for each event.
- `simulated_red_light/`: the whole chain on the simulated signal, with the red-light outputs, and `plates/` with the plate result of each event.

**Plate numbers stay out of git.** A git filter removes every image output from the committed copy of this notebook. Text printed here shows only a plate's first two characters and its length.

## 0. Setup

In [1]:
import json
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from traffic_violation_detection.detection import load_detector
from traffic_violation_detection.plates import (
    NO_PLATE,
    PLATE_SETTINGS,
    READ,
    STATUS_COLORS_BGR,
    UNCERTAIN,
    UNREADABLE,
    best_read,
    combine_reads,
    load_plate_models,
    mask_plate,
    plate_label,
    read_plate_text,
    read_plates_from_clips,
    read_plates_video,
    read_tracks,
    rejection_reason,
    score_plate_results,
)
from traffic_violation_detection.video import iter_frames, read_frame_at, read_video_metadata

%config InlineBackend.figure_formats = ["jpeg"]

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found in {start} or any parent directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DEV_VIDEO = PROJECT_ROOT / "data" / "traffic_video_modified.mp4"
EVAL_VIDEO = PROJECT_ROOT / "data" / "video-test.mp4"
MANIFEST = PROJECT_ROOT / "outputs" / "05b_red_light_violation" / "rule_v2" / "rlr_occasions_violation_clips" / "manifest.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "06_license_plate_recognition"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

labels_path = PROJECT_ROOT / "data" / "private_annotations" / "plate_labels.json"
label_file = json.loads(labels_path.read_text()) if labels_path.exists() else None
dev_labels = label_file["videos"]["traffic_video_modified"]["tracks"] if label_file else {}
eval_section = label_file["videos"]["video-test"] if label_file else {"tracks": {}, "seen_ocr_before_labelling": []}
eval_labels = eval_section["tracks"]
print("labels:", "found" if label_file else "missing, so scoring is skipped", "| reviewed by a person:", label_file["annotator"]["reviewed_by_a_person"] if label_file else None)

# The tracks the development labels were made on. A new tracking run gives slightly different ids, so these are reused.
dev_tracks = {t: rows for t, rows in json.loads((PROJECT_ROOT / "outputs" / "05b_red_light_violation" / "overlay_run.json").read_text())["tracks"].items() if len(rows) >= 10}
print(f"development tracks of at least 10 frames: {len(dev_tracks)}")

plate_models = load_plate_models()
print("models:", PLATE_SETTINGS["detector_model"], "+", PLATE_SETTINGS["ocr_model"])

labels: found | reviewed by a person: False
development tracks of at least 10 frames: 41


models: yolo-v9-t-640-license-plate-end2end + cct-s-v2-global-model


INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


In [3]:
def show(image, title=None, dpi=100):
    height, width = image.shape[:2]
    figure, axis = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.axis("off")
    if title:
        axis.set_title(title, fontsize=11, loc="left")
    plt.show()


def grid(tiles, columns):
    blank = np.zeros_like(tiles[0])
    tiles = tiles + [blank] * (-len(tiles) % columns)
    return np.vstack([np.hstack(tiles[i:i + columns]) for i in range(0, len(tiles), columns)])


def plate_tile(name, result, truth=None, size=(240, 118)):
    # The evidence crop of a result, enlarged, with its status. These images stay in the local copy of this notebook.
    tile = np.zeros((size[1], size[0], 3), np.uint8)
    read = best_read(result["reads"], result["text"])
    if read is not None:
        plate = read["plate_image"]
        scale = min(76 / plate.shape[0], size[0] / plate.shape[1])
        big = cv2.resize(plate, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        tile[:big.shape[0], :big.shape[1]] = big
    color = STATUS_COLORS_BGR[result["status"]]
    cv2.putText(tile, f"{name} {plate_label(result)}", (2, 94), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    if truth is not None:
        cv2.putText(tile, f"label: {truth}", (2, 112), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return tile


def label_text(label):
    return label.get("text") or next((k for k in ("unsure", "unreadable", "no_plate_visible") if k in label), "?")


def print_score(title, score):
    print(title)
    print(f"  labelled readable {score['readable_labels']}: read correctly {score['correct_read']}, read WRONGLY {score['wrong_read']}, right suggestion {score['correct_suggestion']}, wrong suggestion {score['wrong_suggestion']}, declined {score['declined']}, no plate found {score['no_plate']}")
    print(f"  labelled unreadable by eye {score['unreadable_labels']}: invented a read {score['invented_read']}, invented a suggestion {score['invented_suggestion']}")
    print(f"  labels marked unsure, not scored: {score['unsure_label']}")
    if score["wrong_text_edit_distances"]:
        print(f"  characters off in wrong texts: {score['wrong_text_edit_distances']}")


def wilson(successes, total, z=1.96):
    if total == 0:
        return (float("nan"), float("nan"))
    p = successes / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    half = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denominator
    return (max(0.0, center - half), min(1.0, center + half))

## 1. From a vehicle to a plate

A step here is small functions, one per task, joined by a function at the end. These are in `src/traffic_violation_detection/plates.py`:

| Function | Task |
|---|---|
| `load_plate_models` | Load the plate detector and the OCR |
| `select_frames` | Choose the frames where the vehicle looks largest, spread apart |
| `crop_vehicle` | Cut out the vehicle, with a margin, because a box can cut off the plate |
| `pick_vehicle_plate` | Choose this vehicle's plate among those detected |
| `read_plate_text` | Run the OCR on a plate crop |
| `rejection_reason` | Decide whether a single read can be used |
| `read_vehicle_in_frame` | Join the above for one vehicle in one frame |
| `combine_reads` | Turn all of a vehicle's reads into one status and text |
| `read_tracks` | Do this for every vehicle of a video |
| `draw_plate_result`, `write_plate_video` | Draw the result on each frame and save the video |
| `read_plates_video` | **Joins them all, for any video** |
| `read_plates_from_clips` | **Joins them for the clips of candidate violations** |

The tools are a small YOLO plate detector (`open-image-models`) and a transformer OCR trained on plates from many countries (`fast-plate-ocr`), both run with ONNX. Their code is MIT licensed; their READMEs state no license for the weights.

**They run on the GPU, in a process of their own.** In one process with PyTorch, ONNX on the GPU moved PyTorch's later detections by about 0.01 px, enough to change 15 of the 136 tracks of `video-test.mp4`, so one video's result depended on what had run before it. The cause inside one process was not found: two settings of ONNX's convolution search did not help. In a separate process, the detections of all 900 frames stay identical to Notebook 02 and all 136 tracks to Notebook 03, whether YOLO runs before, after or at the same time as the plate models.

**Each vehicle ends in one explicit status:** `read`, `uncertain`, `unreadable` or `no_plate_found`.

**One vehicle, step by step:**

In [4]:
example_id = next((t for t, label in dev_labels.items() if "text" in label and "note" not in label and len(dev_tracks.get(t, [])) >= 30), next(iter(dev_tracks)))
example_results = read_tracks(DEV_VIDEO, {example_id: dev_tracks[example_id]}, *plate_models)
example = example_results[example_id]
read = best_read(example["reads"], example["text"])
frame = read_frame_at(DEV_VIDEO, read["frame"])["image"]
row = next(r for r in dev_tracks[example_id] if r[0] == read["frame"])
x1, y1, x2, y2 = [int(v) for v in row[1:5]]
drawn = frame.copy()
cv2.rectangle(drawn, (x1, y1), (x2, y2), (0, 255, 255), 4)
px1, py1, px2, py2 = read["plate_box_xyxy"]
cv2.rectangle(drawn, (px1, py1), (px2, py2), (0, 255, 0), 4)
plate = cv2.resize(read["plate_image"], None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
small = cv2.resize(drawn, (900, round(drawn.shape[0] * 900 / drawn.shape[1])), interpolation=cv2.INTER_AREA)
canvas = np.zeros((small.shape[0] + plate.shape[0] + 8, small.shape[1], 3), np.uint8)
canvas[:small.shape[0]] = small
canvas[small.shape[0] + 8:small.shape[0] + 8 + plate.shape[0], :plate.shape[1]] = plate[:, :small.shape[1]]
show(canvas, "A vehicle's box (yellow), its plate (green), and the plate crop given to the OCR. Local copy only.")
print(f"track {example_id}: status {example['status']}, text {mask_plate(example['text'])}, {example['votes']} of {example['usable_reads']} usable frames agree, plates found in {example['plates_found']} frames")
print(f"this frame: plate {read['plate_width']}x{read['plate_height']} px, detection confidence {read['detection_confidence']:.2f}, least sure character {read['min_prob']:.2f}")

track 37: status read, text NN***, 4 of 4 usable frames agree, plates found in 5 frames
this frame: plate 172x59 px, detection confidence 0.90, least sure character 1.00


## 2. Choosing the settings, on the development video only

### 2.1 Can one frame be trusted?

Every frame's read is compared with the label. If a single frame's confidence were reliable, none of the rest of this design would be needed.

These numbers use tracks whose label gives a text. Some reads are of a neighbouring vehicle whose plate falls inside the box.

In [5]:
OPEN = {**PLATE_SETTINGS, "min_plate_width": 0, "min_char_prob": 0.0, "min_votes": 1, "min_agreement": 0.0}
dev_open = read_tracks(DEV_VIDEO, dev_tracks, *plate_models, OPEN)
single_reads = [(r, dev_labels[t]["text"]) for t, result in dev_open.items() if "text" in dev_labels.get(t, {}) for r in result["reads"]]
with_text = [(r, truth) for r, truth in single_reads if r["text"]]
print(f"frames with a plate: {len(single_reads)}, of which the OCR gave some text: {len(with_text)}, and an empty text: {len(single_reads) - len(with_text)}")


def correct_by(key, edges):
    for low, high in zip(edges, edges[1:]):
        chosen = [(r, truth) for r, truth in with_text if low <= r[key] < high]
        good = sum(r["text"] == truth for r, truth in chosen)
        print(f"  {low:>5} to {high:<5}: {len(chosen):>3} reads, {good:>3} correct" + (f" ({good / len(chosen):.0%})" if chosen else ""))


print("\nby its least sure character:")
correct_by("min_prob", [0, 0.5, 0.8, 0.95, 0.99, 1.01])
print("by plate width in pixels:")
correct_by("plate_width", [0, 45, 60, 90, 400])

frames with a plate: 119, of which the OCR gave some text: 106, and an empty text: 13

by its least sure character:
      0 to 0.5  :  27 reads,   3 correct (11%)
    0.5 to 0.8  :  21 reads,   5 correct (24%)
    0.8 to 0.95 :   9 reads,   2 correct (22%)
   0.95 to 0.99 :  10 reads,   6 correct (60%)
   0.99 to 1.01 :  39 reads,  33 correct (85%)
by plate width in pixels:
      0 to 45   :   2 reads,   0 correct (0%)
     45 to 60   :  32 reads,   2 correct (6%)
     60 to 90   :  35 reads,  15 correct (43%)
     90 to 400  :  37 reads,  32 correct (86%)


### 2.2 Which models?

Each pairing is scored with the thresholds open, so each track takes its most common text.

In [6]:
print(f"{'detector':<12}{'OCR':<12}{'correct':>8}{'wrong':>7}{'declined':>10}{'seconds':>9}")
for detector_model in ("yolo-v9-t-384-license-plate-end2end", "yolo-v9-t-640-license-plate-end2end", "yolo-v9-s-608-license-plate-end2end"):
    for ocr_model in ("cct-xs-v2-global-model", "cct-s-v2-global-model"):
        settings = {**OPEN, "detector_model": detector_model, "ocr_model": ocr_model}
        models = load_plate_models(settings)
        start = time.perf_counter()
        score = score_plate_results(read_tracks(DEV_VIDEO, dev_tracks, *models, settings), dev_labels)
        print(f"{detector_model[8:13]:<12}{ocr_model[:8]:<12}{score['correct_read']:>8}{score['wrong_read']:>7}{score['declined']:>10}{time.perf_counter() - start:>9.0f}")

detector    OCR          correct  wrong  declined  seconds


INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


t-384       cct-xs-v          13     10         1       10


INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


t-384       cct-s-v2          14     10         0        9


INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


... (3 outputs left out)

INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


s-608       cct-xs-v          13     10         1       11


INFO:open_image_models.detection.core.yolo_v9.inference:Using ONNX Runtime with ['CUDAExecutionProvider', 'CPUExecutionProvider'] provider(s)


s-608       cct-s-v2          13     11         0       13


### 2.3 Does preprocessing help?

The same plate crops are given to the OCR after each treatment, and single reads are counted.

In [7]:
def clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(2.0, (4, 4)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def sharpen(image):
    return cv2.addWeighted(image, 1.6, cv2.GaussianBlur(image, (0, 0), 1.5), -0.6, 0)


def swap_channels(image):
    return image[..., ::-1]


reader = plate_models[1]
crops = [(r["plate_image"], truth) for r, truth in single_reads]
for name, treat in (("none", lambda i: i), ("CLAHE", clahe), ("sharpen", sharpen), ("CLAHE, then sharpen", lambda i: sharpen(clahe(i))), ("wrong colour order, as a check", swap_channels)):
    good = sum(read_plate_text(reader, treat(image))["text"] == truth for image, truth in crops)
    print(f"  {name:<32}{good:>4} of {len(crops)} single reads correct")

  none                              49 of 119 single reads correct


  CLAHE                             50 of 119 single reads correct


  sharpen                           50 of 119 single reads correct


  CLAHE, then sharpen               49 of 119 single reads correct


  wrong colour order, as a check    48 of 119 single reads correct


### 2.4 Which thresholds, and how many frames?

The settings decide what counts as a usable read, and how many frames must agree for a `read`. Each row changes the settings and scores the development tracks again.

In [8]:
def score_settings(**changes):
    # Thresholds can be applied to reads already made.
    settings = {**PLATE_SETTINGS, **changes}
    combined = {}
    for t, result in dev_open.items():
        reads = [{**r, "rejected": rejection_reason(r, settings)} for r in result["reads"]]
        combined[t] = combine_reads(reads, settings)
    return score_plate_results(combined, dev_labels)


def row(name, score):
    print(f"{name:<44}{score['correct_read']:>8}{score['wrong_read']:>7}{score['correct_suggestion']:>9}{score['wrong_suggestion']:>7}{score['declined']:>10}{score['invented_read']:>10}")


header = f"{'':<44}{'correct':>8}{'wrong':>7}{'sug. ok':>9}{'bad':>7}{'declined':>10}{'invented':>10}"
print(header)
row("no thresholds: most common text, one frame", score_settings(min_plate_width=0, min_char_prob=0.0, min_votes=1, min_agreement=0.0))
for prob, votes, agreement in ((0.5, 2, 0.5), (0.5, 2, 0.67), (0.8, 2, 0.5), (0.8, 3, 0.5), (0.95, 2, 0.5), (0.95, 3, 0.5), (0.95, 2, 0.67)):
    mark = "  <- chosen" if (prob, votes, agreement) == (PLATE_SETTINGS["min_char_prob"], PLATE_SETTINGS["min_votes"], PLATE_SETTINGS["min_agreement"]) else ""
    row(f"char >= {prob}, {votes}+ frames, agreement >= {agreement}{mark}", score_settings(min_char_prob=prob, min_votes=votes, min_agreement=agreement))

print("\nframes per track, with the chosen thresholds:")
print(header)
for count, gap in ((6, 4), (12, 4), (24, 4), (48, 2)):
    settings = {**PLATE_SETTINGS, "frames_per_track": count, "min_frame_gap": gap}
    row(f"{count} frames, at least {gap} apart" + ("  <- chosen" if count == PLATE_SETTINGS["frames_per_track"] else ""), score_plate_results(read_tracks(DEV_VIDEO, dev_tracks, *plate_models, settings), dev_labels))

print("\nchosen settings:", {k: v for k, v in PLATE_SETTINGS.items() if k not in ("version", "detector_model", "ocr_model")})

                                             correct  wrong  sug. ok    bad  declined  invented
no thresholds: most common text, one frame        13     10        0      0         1         1
char >= 0.5, 2+ frames, agreement >= 0.5          11      4        2      4         3         0
char >= 0.5, 2+ frames, agreement >= 0.67         10      2        3      6         3         0
char >= 0.8, 2+ frames, agreement >= 0.5           9      4        4      1         6         0
char >= 0.8, 3+ frames, agreement >= 0.5           7      1        6      4         6         0
char >= 0.95, 2+ frames, agreement >= 0.5          8      1        4      4         7         0
char >= 0.95, 3+ frames, agreement >= 0.5          7      1        5      4         7         0
char >= 0.95, 2+ frames, agreement >= 0.67  <- chosen       7      0        5      5         7         0

frames per track, with the chosen thresholds:
                                             correct  wrong  sug. ok    bad  dec

6 frames, at least 4 apart                         2      1        4      4         8         0


12 frames, at least 4 apart  <- chosen             7      0        5      5         7         0


24 frames, at least 4 apart                        8      2        3      4         7         0


48 frames, at least 2 apart                        9      5        1      2         7         0

chosen settings: {'cpu_threads': 8, 'frames_per_track': 12, 'min_frame_gap': 4, 'vehicle_margin': 0.15, 'lowest_plate_center': 0.4, 'min_plate_width': 30, 'min_chars': 4, 'min_char_prob': 0.95, 'min_votes': 2, 'min_agreement': 0.67}


**Why these settings.** A wrong plate could point at the wrong person, so the choice favours **zero wrong reads** on this video, at the cost of declining more often. A `read` needs frames that agree, each with every character at 0.95 or more.

These settings were chosen on one video with 24 readable tracks, many of them the same few vehicles, so they could easily be overfitted. Section 3 tests them on another video.

**The chosen settings on the development video:**

In [9]:
dev_results = read_tracks(DEV_VIDEO, dev_tracks, *plate_models)
print("statuses:", dict(Counter(r["status"] for r in dev_results.values())))
print_score("development video, chosen settings", score_plate_results(dev_results, dev_labels))
tiles = [plate_tile(f"t{t}", dev_results[t], label_text(label)) for t, label in dev_labels.items() if t in dev_results]
if tiles:
    show(grid(tiles, 5), "Development video: the crop kept for each track, its result, and the label. Local copy only.")

statuses: {'uncertain': 10, 'read': 7, 'unreadable': 12, 'no_plate_found': 12}
development video, chosen settings
  labelled readable 24: read correctly 7, read WRONGLY 0, right suggestion 5, wrong suggestion 5, declined 7, no plate found 0
  labelled unreadable by eye 2: invented a read 0, invented a suggestion 0
  labels marked unsure, not scored: 3
  characters off in wrong texts: [1, 1, 1, 1, 6]


## 3. Evaluation on another video

`video-test.mp4` is daytime, in another country, with two-line motorcycle plates. Its labels were made from crops showing no OCR output, and the settings were committed before it was run with them.

**One exception:** an earlier first look showed the OCR's output for 18 of these tracks before they were labelled, so the score is also given without them.

In [10]:
detector = load_detector()
eval_run = read_plates_video(EVAL_VIDEO, OUTPUT_DIR / "video-test", model=detector, plate_models=plate_models)
eval_results = eval_run["results"]
print("tracks:", len(eval_results), "| statuses:", dict(Counter(r["status"] for r in eval_results.values())))

# The labels use the track ids of Notebook 03, so the ids must match them.
baseline = json.loads((PROJECT_ROOT / "outputs" / "03_multi_object_tracking" / "tracks" / "video-test.ocsort-byte.trajectories.json").read_text())["tracks"]
same = sum(t in eval_run["tracks"] and [r[0] for r in eval_run["tracks"][t]] == track["frames"] for t, track in baseline.items())
print(f"tracks identical to Notebook 03: {same} of {len(baseline)}")
assert same == len(baseline)

eval_score = score_plate_results(eval_results, eval_labels)
print_score("\nevaluation video, all labelled tracks", eval_score)
seen = set(eval_section["seen_ocr_before_labelling"])
print_score("\nevaluation video, without the 18 tracks seen earlier", score_plate_results(eval_results, {t: l for t, l in eval_labels.items() if t not in seen}))

good, bad = eval_score["correct_read"], eval_score["wrong_read"]
low, high = wilson(good, good + bad)
if good + bad:
    print(f"\nA 'read' is correct: {good} of {good + bad} = {good / (good + bad):.2f} (95% interval {low:.2f}-{high:.2f})")
print(f"A readable plate gets a correct 'read': {good} of {eval_score['readable_labels']}" + (f" = {good / eval_score['readable_labels']:.2f}" if eval_score["readable_labels"] else ""))
print(f"Labelled tracks hold {len({l['text'] for l in eval_labels.values() if 'text' in l})} different plates: several tracks can be one vehicle.")
tiles = [plate_tile(f"t{t}", eval_results.get(t, {"status": NO_PLATE, "text": None, "reads": []}), label_text(label)) for t, label in eval_labels.items()]
if tiles:
    show(grid(tiles, 5), "Evaluation video: the crop kept for each track, its result, and the label. Local copy only.")

tracks: 136 | statuses: {'unreadable': 84, 'uncertain': 4, 'no_plate_found': 42, 'read': 6}


tracks identical to Notebook 03: 136 of 136

evaluation video, all labelled tracks
  labelled readable 14: read correctly 6, read WRONGLY 0, right suggestion 3, wrong suggestion 0, declined 3, no plate found 2
  labelled unreadable by eye 35: invented a read 0, invented a suggestion 0
  labels marked unsure, not scored: 11

evaluation video, without the 18 tracks seen earlier
  labelled readable 7: read correctly 0, read WRONGLY 0, right suggestion 2, wrong suggestion 0, declined 3, no plate found 2
  labelled unreadable by eye 28: invented a read 0, invented a suggestion 0
  labels marked unsure, not scored: 7

A 'read' is correct: 6 of 6 = 1.00 (95% interval 0.61-1.00)
A readable plate gets a correct 'read': 6 of 14 = 0.43
Labelled tracks hold 9 different plates: several tracks can be one vehicle.


**A cost found on this video.** Two readable plates ended as `no_plate_found`, although the detector had found plates on those tracks, so `pick_vehicle_plate` rejected them. One track's plate is above its box. The other is a motorcycle whose plate's center lies at about 35% of the box height, and the rule requires 40%. That limit was meant to reject roof signs on cars, and it had no measurable effect on the development video.

Changing it now would make this no longer a fair test, so **the result above stands**. This cell only shows what removing the limit would do, for a next version:

In [11]:
without_limit = read_tracks(EVAL_VIDEO, eval_run["tracks"], *plate_models, {**PLATE_SETTINGS, "lowest_plate_center": 0.0})
print("statuses without the limit:", dict(Counter(r["status"] for r in without_limit.values())))
print_score("evaluation video, without the limit (not adopted)", score_plate_results(without_limit, eval_labels))

statuses without the limit: {'unreadable': 96, 'uncertain': 4, 'no_plate_found': 29, 'read': 7}
evaluation video, without the limit (not adopted)
  labelled readable 14: read correctly 7, read WRONGLY 0, right suggestion 3, wrong suggestion 0, declined 3, no plate found 1
  labelled unreadable by eye 35: invented a read 0, invented a suggestion 0
  labels marked unsure, not scored: 11


**How small a plate can it read?** Plates that were read correctly are shrunk to each width and read again. This asks what a camera would need to deliver, using sharp crops; real small plates are also blurred, so this is an upper limit.

In [12]:
sharp = []
for results, labels in ((dev_results, dev_labels), (eval_results, eval_labels)):
    for t, result in results.items():
        if "text" in labels.get(t, {}) and result["text"] == labels[t]["text"]:
            read = best_read(result["reads"], result["text"])
            if read["plate_width"] >= 60:
                sharp.append((read["plate_image"], result["text"]))
print(f"{len(sharp)} crops at least 60 px wide that were read correctly")
for width in (100, 80, 60, 45, 35, 25, 18):
    good = 0
    for image, truth in sharp:
        scale = min(1.0, width / image.shape[1])
        small = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        good += read_plate_text(reader, small)["text"] == truth
    print(f"  shrunk to {width:>3} px wide: {good:>2} of {len(sharp)} still correct")

14 crops at least 60 px wide that were read correctly
  shrunk to 100 px wide: 14 of 14 still correct
  shrunk to  80 px wide: 14 of 14 still correct
  shrunk to  60 px wide: 14 of 14 still correct


  shrunk to  45 px wide: 14 of 14 still correct


  shrunk to  35 px wide: 13 of 14 still correct
  shrunk to  25 px wide:  7 of 14 still correct
  shrunk to  18 px wide:  0 of 14 still correct


## 4. The clips of the candidate violations

This is the route real cases take: the clips and `manifest.json` from Notebook 05b. Each clip's vehicle is known from the manifest, so no detector or tracker runs here. Every event gets a `license_plate` result, including an explicit statement when nothing can be read.

In [13]:
clips_run = read_plates_from_clips(MANIFEST, OUTPUT_DIR / "violation_clips", plate_models=plate_models)
print("statuses:", dict(Counter(e["license_plate"]["status"] for e in clips_run["events"])))
print(f"\n{'event':<44}{'status':<16}{'text':<12}{'found':>6}{'usable':>7}{'largest plate':>15}")
for event in clips_run["events"]:
    plate = event["license_plate"]
    sizes = [(r["plate_width"], r["plate_height"]) for r in plate["reads"]]
    largest = max(sizes, default=None)
    print(f"{event['event_id']:<44}{plate['status']:<16}{str(mask_plate(plate['text'])):<12}{plate['plates_found']:>6}{plate['usable_reads']:>7}{str(largest):>15}")
tiles = [plate_tile(event_id.split('-red_light-')[0].split('-')[-1] + " " + event_id.split('-')[-1], result) for event_id, result in clips_run["results"].items()]
show(grid(tiles, 4), "Violation clips: the crop kept for each event, and its result. Local copy only.")

statuses: {'unreadable': 14, 'no_plate_found': 3}

event                                       status          text         found usable  largest plate
evt-rlr_occasions-A-red_light-f00153-t12    unreadable      None            12      0       (38, 20)
evt-rlr_occasions-B-red_light-f00718-t23    unreadable      None             4      0        (25, 9)
evt-rlr_occasions-C-red_light-f01027-t1     no_plate_found  None             0      0           None
evt-rlr_occasions-C-red_light-f01059-t2     unreadable      None             4      0       (38, 15)
evt-rlr_occasions-D-red_light-f01166-t12    unreadable      None             7      0       (24, 16)
evt-rlr_occasions-E-red_light-f01588-t0     unreadable      None             2      0       (22, 11)
evt-rlr_occasions-F-red_light-f01761-t2     unreadable      None             1      0        (18, 6)
evt-rlr_occasions-G-red_light-f02033-t1     unreadable      None             4      0       (40, 24)
evt-rlr_occasions-J-red_light-f03248-t0 

## 5. From any video to a video with its plates

The function that joins everything, on the development video:

```python
from traffic_violation_detection.plates import read_plates_video

result = read_plates_video("path/to/video.mp4", "path/to/output_folder")
```

It saves `<video name>_output.mp4`, `<video name>_plate_crops/` and `<video name>_plates.json`. The video shows each vehicle's result, with a question mark on a text that is only a suggestion. The video's signal is a graphic, and nothing here claims a violation.

In [14]:
start = time.perf_counter()
dev_run = read_plates_video(DEV_VIDEO, OUTPUT_DIR / "traffic_video_modified", model=detector, plate_models=plate_models)
elapsed = time.perf_counter() - start
source_meta, output_meta = read_video_metadata(DEV_VIDEO), read_video_metadata(dev_run["output_video"])
frames_written = sum(1 for _ in iter_frames(dev_run["output_video"]))
print(f"{dev_run['output_video'].relative_to(PROJECT_ROOT)}: {dev_run['output_video'].stat().st_size / 1e6:.1f} MB, {elapsed:.0f} s for {frames_written} frames")
print(f"input : {source_meta['width']}x{source_meta['height']}, {source_meta['fps']:.2f} FPS, {source_meta['reported_frame_count']} frames")
print(f"output: {output_meta['width']}x{output_meta['height']}, {output_meta['fps']:.2f} FPS, {frames_written} frames")
print("statuses:", dict(Counter(r["status"] for r in dev_run["results"].values())))
assert frames_written == source_meta["reported_frame_count"]
frame = read_frame_at(dev_run["output_video"], 300)["image"]
show(cv2.resize(frame, (900, round(frame.shape[0] * 900 / frame.shape[1])), interpolation=cv2.INTER_AREA), "A frame of the output video. Local copy only.")

outputs/06_license_plate_recognition/traffic_video_modified/traffic_video_modified_output.mp4: 27.5 MB, 41 s for 715 frames
input : 1920x1012, 26.33 FPS, 715 frames
output: 1920x1012, 26.33 FPS, 715 frames
statuses: {'uncertain': 10, 'read': 7, 'unreadable': 12, 'no_plate_found': 16}


## 6. Validation

**Missing information.** Images with nothing to read must not become a result, however sure the OCR is.

In [15]:
rng = np.random.default_rng(0)
cases = {
    "uniform gray": np.full((40, 80, 3), 128, np.uint8),
    "black": np.zeros((40, 80, 3), np.uint8),
    "white": np.full((40, 80, 3), 255, np.uint8),
    "noise": rng.integers(0, 256, (40, 80, 3), dtype=np.uint8),
    "blurred noise": cv2.GaussianBlur(rng.integers(0, 256, (40, 80, 3), dtype=np.uint8), (0, 0), 6),
}
print(f"{'image':<16}{'reading':<14}{'least sure':>11}  rejected as")
rejected_all = True
for name, image in cases.items():
    read = {**read_plate_text(reader, image), "plate_width": image.shape[1]}
    reason = rejection_reason(read)
    rejected_all &= reason is not None
    print(f"{name:<16}{'empty' if not read['text'] else 'some text':<14}{read['min_prob']:>11.2f}  {reason}")
assert rejected_all

image           reading        least sure  rejected as
uniform gray    empty                0.00  too_short
black           empty                0.00  too_short
white           empty                0.00  too_short
noise           empty                0.00  too_short
blurred noise   empty                0.00  too_short


**Data contract.** Every result must name its vehicle or event, carry an explicit status, and keep its crop as evidence when one exists. Events stay `pending`.

In [16]:
for path, key in ((eval_run["plates_file"], "tracks"), (dev_run["plates_file"], "tracks")):
    saved = json.loads(path.read_text())
    for track_id, result in saved[key].items():
        assert result["status"] in (READ, UNCERTAIN, UNREADABLE, NO_PLATE)
        assert (result["text"] is None) == (result["status"] in (UNREADABLE, NO_PLATE))
        assert (result["plate_crop"] is None) == (result["plates_found"] == 0)
saved = json.loads(clips_run["plates_file"].read_text())
assert len(saved["events"]) == len(json.loads(MANIFEST.read_text())["clips"])
assert all(e["review_status"] == "pending" and e["license_plate"]["status"] in (READ, UNCERTAIN, UNREADABLE, NO_PLATE) for e in saved["events"])
print("data contract holds for", len(json.loads(eval_run["plates_file"].read_text())["tracks"]), "+", len(json.loads(dev_run["plates_file"].read_text())["tracks"]), "tracks and", len(saved["events"]), "events")

data contract holds for 136 + 45 tracks and 17 events


## 7. The whole chain, on a simulated signal

The violation clips of section 4 are too blurred to read, so they cannot show the whole chain working. This section simulates a real case: the signal graphic of `traffic_video_modified.mp4` is **treated as the real signal of the road**. Its plates can be read, so one run goes from a video to a plate:

`detect_red_light_video` → one clip per candidate → `read_plates_from_clips` → a plate result for each event

The scene configuration `configs/scenes/traffic_video_modified.json` declares the painted stop line, the lamps of the graphic, the moving camera, and traffic that comes toward the camera. These needed three additions to the red-light code, all off by default, and Notebook 05b's rule version 1 still gives the same 16 events. Events stay `pending`, and carry `signal_source: simulated_overlay`, which keeps them traceable.

In [17]:
from traffic_violation_detection.red_light import CANDIDATE, detect_red_light_video

SIM_DIR = OUTPUT_DIR / "simulated_red_light"
SIM_SCENE = PROJECT_ROOT / "configs" / "scenes" / "traffic_video_modified.json"
sim_run = detect_red_light_video(DEV_VIDEO, SIM_DIR, SIM_SCENE, model=detector)
sim_crossings = sim_run["crossings"]["main"]
sim_fps = read_video_metadata(DEV_VIDEO)["fps"]

print(f"{'frame':>6}{'time s':>8}  {'signal':<8}{'class':<7}verdict")
for r in sim_crossings:
    print(f"{r['frame']:>6}{r['frame'] / sim_fps:>8.2f}  {r['signal_state']:<8}{r['class_name']:<7}{r['verdict']}")
print("\ncrossings by signal:", dict(Counter(r["signal_state"] for r in sim_crossings)), "| candidates:", len(sim_run["events"]))
assert all(r["signal_state"] == "red" for r in sim_crossings if r["verdict"] == CANDIDATE)
assert all(e["review_status"] == "pending" and e["signal_source"] == "simulated_overlay" for e in sim_run["events"])

 frame  time s  signal  class  verdict
    50    1.90  green   car    signal_not_red
   102    3.87  green   car    signal_not_red
   145    5.51  green   car    signal_not_red
   154    5.85  green   car    signal_not_red
   220    8.35  conflictcar    not_evaluable_signal
   238    9.04  yellow  car    signal_not_red
   268   10.18  yellow  bus    signal_not_red
   317   12.04  red     car    candidate
   365   13.86  red     car    candidate
   417   15.84  green   car    signal_not_red
   624   23.70  red     truck  stopped_at_line

crossings by signal: {'green': 5, 'conflict': 1, 'yellow': 2, 'red': 3} | candidates: 2


In [18]:
sim_plates = read_plates_from_clips(sim_run["clips_manifest"], SIM_DIR / "plates", plate_models=plate_models)
sim_labels = label_file.get("simulated_red_light", {}) if label_file else {}
tiles = []
for event, plate_event in zip(sim_run["events"], sim_plates["events"]):
    plate = plate_event["license_plate"]
    truth = sim_labels.get(str(event["trigger_frame"]))
    print(f"{event['event_id']}\n  status {plate['status']}, text {mask_plate(plate['text'])}, {plate['votes']} of {plate['usable_reads']} usable frames agree, plates found in {plate['plates_found']} frames, same as the label: {plate['text'] == truth if truth else 'no label'}")
    result = sim_plates["results"][event["event_id"]]
    frame = read_frame_at(sim_run["output_video"], event["trigger_frame"])["image"]
    tiles.append(cv2.resize(frame, (640, round(frame.shape[0] * 640 / frame.shape[1])), interpolation=cv2.INTER_AREA))
    crop = plate_tile("", result, size=(640, tiles[-1].shape[0]))
    tiles.append(crop)
if tiles:
    show(grid(tiles, 2), "The candidate at the moment of entry, and its plate. Local copy only.")

evt-traffic_video_modified-main-red_light-f00317-t23
  status read, text YB****, 9 of 9 usable frames agree, plates found in 10 frames, same as the label: True
evt-traffic_video_modified-main-red_light-f00365-t37
  status read, text NN***, 7 of 7 usable frames agree, plates found in 8 frames, same as the label: True


**Checked by eye, every vehicle in both red phases.** All verdicts on red are right. Besides the flagged cars and a truck that stops at the line:

- **The yellow car is found only by rule version 2.** The first flagged car hid it, its track ended before the line at frame 350, and a new track began past the line at frame 389. No track crosses the line, so its entry is estimated. The estimate, frame 365, is about 20 frames earlier than the crossing seen by eye at about frame 386, which is why the signal must hold for the whole window: red lasted from before the estimate until the car was seen.
- **A blue car, at about the moment the signal turned red, cannot be judged.** A taxi hid it as it crossed, and it is first seen past the line. It gets no estimate, so it is not flagged.
- **Another car, hidden as the signal changed from green to yellow, is `not_evaluable_signal`,** as its estimated window covers both states.

## Results

All numbers come from the outputs above. Labels were made by the AI assistant that wrote this code, and **no person has reviewed them**.

**On the evaluation video**, with settings committed before it ran:

| Tracks labelled readable (14) | |
|---|---|
| `read`, correct | 6 |
| `read`, **wrong** | **0** |
| `uncertain`, with the right text | 3 |
| `uncertain`, with a wrong text | 0 |
| `unreadable` | 3 |
| `no_plate_found` | 2 |

- **When it says `read`, it was right:** 6 of 6, for 5 different plates (95% interval 0.61–1.00).
- **It reads less than half of what a person can:** 6 of 14 tracks (0.43).
- **It invented nothing:** none of the 35 tracks unreadable by eye got a text.

**This does not independently confirm the reads.** All 6 are among the 18 tracks whose OCR output was seen before they were labelled, which are also the largest plates. On the 7 readable tracks never seen beforehand, there were no reads at all: 2 right suggestions, 3 `unreadable` and 2 `no_plate_found`.

**On the development video**, where the settings were chosen, so this is optimistic: of 24 readable tracks, 7 correct reads, 0 wrong, 5 right suggestions, **5 wrong suggestions**, and 7 `unreadable`. A text marked `uncertain` is therefore only a hint.

**On the 17 clips of candidate violations: no plate was read.** 14 are `unreadable` and 3 `no_plate_found`. The largest plate found is 40×24 px, and the crops are blurred or flared by tail lights. Each event states this explicitly, which is the required behaviour, but **these clips cannot link a violation to a plate**.

**On the simulated signal, the whole chain works.** With the graphic treated as the real signal, rule version 2 judged 11 crossings and flagged 2 cars on red. Both plates were `read` from their clips, with 9 of 9 and 7 of 7 usable frames agreeing, and both match the plates read by eye. One of the two was hidden at the line, so its entry is estimated.

**What the design choices rest on:**

- **One frame's confidence is not enough.** Reads with every character at 0.99 or more are right 85% of the time (33 of 39), so some wrong ones are fully confident.
- **More frames are not always better.** With 12 frames there were 0 wrong reads, and with 48 frames there were 5. A consistent OCR error also collects more votes.
- **Blur limits reading more than size.** Night plates 45–60 px wide were read correctly 6% of the time. Sharp crops shrunk to 45 px stayed correct in 14 of 14 cases, to 35 px in 13, and to 25 px in 7. Readable footage needs plates that are both sharp and at least about 35–45 px wide.
- **Preprocessing was not adopted.** The best treatments gave 50 correct single reads of 119, against 49 with none.
- **Colour order showed no effect.** The OCR's documented input is RGB, and the code converts to it, but swapping the channels gave 48 of 119, against 49.
- **The models made no clear difference.** All six pairings gave 12–14 correct and 10–11 wrong with thresholds open.
- **The models run on the GPU in a process of their own**, so that results do not depend on what ran before, as explained in section 1.

## False positives and false negatives

- **Wrong reads:** none at the chosen settings, on either video. Looser settings do produce them, so this rests on the strict thresholds and on a small sample.
- **Track errors, not OCR errors:** a track can jump to another vehicle. In the simulated case, the flagged car's track later moves to other vehicles, so `read_plates_video`, which reads whole tracks, gives another vehicle's plate for it. The event's clip ends 2 s after entry, so `read_plates_from_clips` reads the right car. At 48 frames per track, 2 of the 5 wrong reads have this cause.
- **Consistent errors:** the OCR can make the same mistake in many frames, such as dropping a repeated digit from a two-line plate. Frames that agree cannot reveal this. At the chosen settings these mostly end as `unreadable` or as a wrong suggestion, because of the strict confidence.
- **A neighbour's plate:** a box covering two vehicles can yield the other vehicle's plate, and with more frames this became a wrong `read`. The chosen settings leave such a track `uncertain`, but this comes from tracking and can return.
- **Missed plates:** the 3 `unreadable` tracks were readable by eye, and were declined by the strict thresholds.
- **A rule that cost a plate:** `pick_vehicle_plate` requires a plate's center below 40% of the box height. That rejected at least one real motorcycle plate, whose center is at about 35%. Removing the limit gives 7 correct reads with 0 wrong, and finds plates on 13 more tracks. This was seen on the evaluation video, so it has not been adopted, and that result is not a fair test of the change.

## Limitations

- **Few plates, labelled by the author.** The evaluation has 14 readable tracks with 9 different plates, and the development video has 24 tracks with 12. Plates that a person cannot read have no truth to compare with.
- **The settings may be overfitted** to one night video, and only one other video has tested them.
- **Plate systems tested:** two-line and one-line plates from two regions. Plates in the violation clips were too small to test at all. No format check is applied, because the OCR is meant to work across countries.
- **Only 60 tracks of the evaluation video are labelled**, those with the largest plates.
- **Weights' license.** The code is MIT licensed, but the READMEs state no license for the model weights, which should be checked before any use beyond study.
- **Speed.** About 11 plates per second on an RTX 4090, including decoding the frames.

## Conclusion

**Completion criterion** — *the notebook can process selected vehicle imagery and return structured license plate results, including an explicit unknown/unreadable state*:

- **Met.** Every vehicle and event gets one of four explicit statuses, with its reads, confidence and evidence crop. Images with nothing to read are rejected, and every event stays `pending`.
- **Accuracy is not yet established.** No wrong read has occurred, but the supporting sample is small and was not blind. Plates on the real violation clips cannot be read at all.

**Readiness for integration.** The small functions and the two joining functions are in `src/traffic_violation_detection/plates.py` with tests. `violation_clips/plates.json` gives a `license_plate` result per event, which is the input to Notebook 07.

**Next steps.**

- **Footage with readable plates and real violations** matters most: the whole chain has worked only on a simulated signal, and for two vehicles. Plates should be sharp and at least about 35–45 px wide.
- **A person's review** of the plate labels, alongside the red-light labels.
- **Remove the plate height limit** in a next version, and test it on new footage.
- **Notebook 07 — event generation** can proceed, using the explicit statuses from here.